In [1]:
import os
import pandas as pd

In [2]:
file_path = os.path.join(os.getcwd(), "balanced-reviews.txt")
df = pd.read_csv(file_path, sep="\t", encoding="utf-16")

In [3]:
df.head()

,no,Hotel name,rating,user type,room type,nights,review
0,2,فندق 72,2,مسافر منفرد,غرفة ديلوكس مزدوجة أو توأم,أقمت ليلة واحدة,“ممتاز”. النظافة والطاقم متعاون.
1,3,فندق 72,5,زوج,غرفة ديلوكس مزدوجة أو توأم,أقمت ليلة واحدة,استثنائي. سهولة إنهاء المعاملة في الاستقبال. ل...
2,16,فندق 72,5,زوج,-,أقمت ليلتين,استثنائي. انصح بأختيار الاسويت و بالاخص غرفه ر...
3,20,فندق 72,1,زوج,غرفة قياسية مزدوجة,أقمت ليلة واحدة,“استغرب تقييم الفندق كخمس نجوم”. لا شي. يستحق ...
4,23,فندق 72,4,زوج,غرفة ديلوكس مزدوجة أو توأم,أقمت ليلتين,جيد. المكان جميل وهاديء. كل شي جيد ونظيف بس كا...


In [4]:
df.columns

Index(['no', 'Hotel name', 'rating', 'user type', 'room type', 'nights',
       'review'],
      dtype='object')

In [5]:
df.isna().sum()

no            0
Hotel name    0
rating        0
user type     0
room type     0
nights        0
review        0
dtype: int64

In [6]:
df[["review"]]

,review
0,“ممتاز”. النظافة والطاقم متعاون.
1,استثنائي. سهولة إنهاء المعاملة في الاستقبال. ل...
2,استثنائي. انصح بأختيار الاسويت و بالاخص غرفه ر...
3,“استغرب تقييم الفندق كخمس نجوم”. لا شي. يستحق ...
4,جيد. المكان جميل وهاديء. كل شي جيد ونظيف بس كا...
...,...
105693,“فند”. لا شئ عجبني. طقم العمل سيئ جدالا يوجد ب...
105694,“سيئ”. قربه من المسجد النبوي الشريف. استخدام م...
105695,“اسوأ إقامة في الرحلة !”. القرب من الحرم. كل ش...
105696,“دون المستوى”. قربه من الحرم. كل شيء


In [ ]:
from collections import Counter
import pandas as pd
import re

arabic_stopwords = set([
    "في", "من", "و", "على","علي","الي" ,"الى", "إلى", "عن", "مع", "هذا", "هذه", "كان", "كانت",
    "كل", "لا", "ما", "لم", "لن", "هو", "هي", "هم", "ان", "أن", "إن", "او", "أو",
    "ثم", "حتى", "بعد", "قبل", "عند", "غير", "بدون", "فقط", "جدا", "جداً", "شيء",
    "شي", "اي", "أي", "بس", "ولا", "لو", "لي", "له", "لها", "بها", "فيه", "فيها",
    "الذي", "التي", "اللي", "عشان", "لكن", "كمان", "انا", "أنه", "انه", "حيث", "تم"
])

reviews_series = df[df["rating"] <= 2]["review"].dropna()


words = []
for text in reviews_series:

    text = re.sub(r"[إأآا]", "ا", str(text))
    text = re.sub(r"ة", "ه", text)
    text = re.sub(r"ى", "ي", text)
    text = re.sub(r"[^\w\s]", " ", text)
    
    for word in text.split():

        if len(word) > 2 and word not in arabic_stopwords:
            words.append(word)


top_words = Counter(words).most_common(20)
top_words_df = pd.DataFrame(top_words, columns=["الكلمة (Word)", "عدد التكرار (Count)"])

display(top_words_df)

,الكلمة (Word),عدد التكرار (Count)
0,الفندق,21781
1,ضعيف,13707
2,الغرفه,11056
3,الاستقبال,10302
4,مخيب,9744
5,الغرف,9650
6,للامل,9576
7,سيئه,9360
8,يوجد,7998
9,فندق,7863


In [ ]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer(
    ngram_range=(2, 2), 
    stop_words=list(arabic_stopwords), 
    max_features=20
)

X = vectorizer.fit_transform(reviews_series)
counts = X.sum(axis=0).A1
bigrams = vectorizer.get_feature_names_out()

bigrams_df = pd.DataFrame({"العبارة (Bigram)": bigrams, "التكرار (Count)": counts})
bigrams_df = bigrams_df.sort_values(by="التكرار (Count)", ascending=False).reset_index(drop=True)

display(bigrams_df)

,العبارة (Bigram),التكرار (Count)
0,مخيب للأمل,9544
1,موظف الاستقبال,1401
2,موقع الفندق,1343
3,عدم وجود,1269
4,طاقم العمل,1192
5,الواي فاي,1188
6,انصح به,1063
7,للأمل الموقع,1006
8,خدمة الغرف,823
9,الاثاث قديم,804
